In [1]:
import json
import os
import re

import data_utils
from concept_generation_qwen import LocalQwenGenerator

In [17]:
dataset = "esc50"
prompt_type = "around"  # important | superclass | around

model_id = "Qwen/Qwen3.5-9B"
device = "cuda"

num_trials = 2
max_new_tokens = 256
temperature = 0.7
top_p = 0.9

In [18]:
prompts = {
    "important": """You are an expert in sound perception.

Task: List 3-5 of the most important auditory concepts that help recognize the sound class "{class}".

Requirements:
- Concepts must be directly audible (no visual or semantic descriptions)
- Use concise, interpretable phrases (1-3 words)
- Prefer informative multi-word concepts when useful (e.g., "rapid barking", "steady rainfall")
- Single words are allowed if they are clear and meaningful
- Focus on discriminative cues (what makes this class distinct)
- Avoid vague terms (e.g., "noise", "sound")
- Avoid duplicates or synonyms

Examples:

Class: dog
- loud barking
- low growling
- rapid panting
- sharp whining
- distant howling

Class: rain
- steady rainfall
- light dripping
- irregular splashing
- distant thunder
- wind noise

Now list 3-5 of the most important auditory concepts for "{class}":
"""
    ,
    "superclass": """You are an expert in sound categorization.

Task: Provide 3-5 of superclass categories for the sound class "{class}".

Requirements:
- Use short, interpretable labels (1-3 words)
- Prefer meaningful phrases over single generic words
- Return 2-5 categories
- Include multiple abstraction levels (specific → general)
- Categories must describe types of sounds, not contexts
- Avoid redundancy

Examples:

Class: dog
- animal vocalization
- mammal sound
- domestic animal sound
- biological sound source

Class: rain
- weather sound
- natural ambient sound
- environmental soundscape
- atmospheric sound

Now provide 3-5 superclasses for "{class}":
"""
    ,
"around": """You are analyzing real-world audio recordings.

Task: List 3-5 of the sounds that commonly occur together with the sound class "{class}".

Requirements:
- Do NOT include the main class itself
- Use concise, interpretable phrases (1-3 words)
- Prefer informative multi-word concepts (e.g., "distant traffic noise", "human conversation")
- Focus on clearly audible elements in the environment
- Avoid vague terms (e.g., "background noise")
- Avoid duplicates

Examples:

Class: dog
- human speech
- footsteps on ground
- door opening sounds
- leash movement noise
- park ambience

Class: rain
- wind noise
- distant traffic
- thunder rumble
- flowing water
- umbrella movement

Now list 3-5 sounds commonly heard around "{class}":
""" }

base_prompt = prompts[prompt_type]
base_prompt[:300]

'You are analyzing real-world audio recordings.\n\nTask: List 3-5 of the sounds that commonly occur together with the sound class "{class}".\n\nRequirements:\n- Do NOT include the main class itself\n- Use concise, interpretable phrases (1-3 words)\n- Prefer informative multi-word concepts (e.g., "distant tr'

In [19]:
classes = data_utils.get_dataset_classes(dataset)

save_dir = "data/concept_sets/qwen_init"
os.makedirs(save_dir, exist_ok=True)
save_path = "{}/qwen_{}_{}.json".format(save_dir, dataset, prompt_type)

len(classes), classes[:10], save_path

(50,
 ['dog',
  'rooster',
  'pig',
  'cow',
  'frog',
  'cat',
  'hen',
  'insects',
  'sheep',
  'crow'],
 'data/concept_sets/qwen_init/qwen_esc50_around.json')

In [5]:
generator = LocalQwenGenerator(model_id=model_id, device=device)
"generator ready"

'generator ready'

In [22]:
generator.generate(
    base_prompt.format(**{"class": "Dinosaur"}),
    max_new_tokens=32,
    temperature=0.2,
    top_p=0.9,
    enable_thinking=False,
 )

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

'- heavy breathing\n- low growl\n- tail swish\n- foot stomps\n- leaf rustle\n- distant thunder\n- human whispers\n'

In [20]:
feature_dict = {}
meta_terms = ["analyze", "constraint", "task", "goal", "thinking", "reasoning", "instructions", "output format"]

for i, label in enumerate(classes):
    feature_dict[label] = set()
    print("\n", i, label)
    for _ in range(num_trials):
        try:
            formatted_prompt = base_prompt.format(**{"class": label})
        except (KeyError, IndexError):
            formatted_prompt = base_prompt.format(label)

        response = generator.generate(
            formatted_prompt,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            enable_thinking=False,
        )

        features = response.split("\n")
        cleaned = []
        for feat in features:
            feat = re.sub(r"^[-*\\d\\.)\\s]+", "", feat).strip()
            feat = feat.strip('"').strip("'").strip().strip(",")
            if len(feat) == 0:
                continue

            lower_feat = feat.lower()
            if any(term in lower_feat for term in meta_terms):
                continue
            if len(feat.split()) > 6:
                continue

            cleaned.append(feat)

        feature_dict[label].update(cleaned)

    feature_dict[label] = sorted(list(feature_dict[label]))

len(feature_dict), len(next(iter(feature_dict.values()))) if len(feature_dict) > 0 else 0


 0 dog

 1 rooster

 2 pig

 3 cow

 4 frog

 5 cat

 6 hen

 7 insects

 8 sheep

 9 crow

 10 rain

 11 sea_waves

 12 crackling_fire

 13 crickets

 14 chirping_birds

 15 water_drops

 16 wind

 17 pouring_water

 18 toilet_flush

 19 thunderstorm

 20 crying_baby

 21 sneezing

 22 clapping

 23 breathing

 24 coughing

 25 footsteps

 26 laughing

 27 brushing_teeth

 28 snoring

 29 drinking_sipping

 30 door_wood_knock

 31 mouse_click

 32 keyboard_typing

 33 door_wood_creaks

 34 can_opening

 35 washing_machine

 36 vacuum_cleaner

 37 clock_alarm

 38 clock_tick

 39 glass_breaking

 40 helicopter

 41 chainsaw

 42 siren

 43 car_horn

 44 engine

 45 train

 46 church_bells

 47 airplane

 48 fireworks

 49 hand_saw


(50, 7)

In [21]:
with open(save_path, "w", encoding="utf-8") as outfile:
    json.dump(feature_dict, outfile, indent=4, ensure_ascii=True)

save_path

'data/concept_sets/qwen_init/qwen_esc50_around.json'

In [22]:
with open(save_path, "r", encoding="utf-8") as f:
    saved = json.load(f)

example_label = classes[1]
example_label, saved[example_label], len(saved)

('rooster',
 ['crowing chorus',
  'crowing roosters',
  'distant farm machinery',
  'early morning birds',
  'human conversation',
  'human footsteps',
  'wind rustling leaves'],
 50)

In [16]:
"done"

'done'